In this notebook we will sample a further 4% of data to be used as training in the same way as before, and combine them to get a total of 9%, with the same validation data.

In [1]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

# --------------------------------------------------
# PATHS
# --------------------------------------------------

SPECTRA_DIR = Path(
    "spectra_train"
)

PREVIOUS_DATA_DIR = Path("model_datasets/stage_1_5pct")

OUTPUT_DIR = Path("model_datasets/stage_2_9pct")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# CONFIGURATION
# --------------------------------------------------

SEED = 42
ADDITIONAL_SAMPLE_FRACTION = 0.05
TRAIN_FRACTION = 0.80

PART_FILES = [
    f"part-{i:03d}.parquet"
    for i in range(21)
]

print("Configuration loaded.")
print(f"Number of source files: {len(PART_FILES)}")
print(f"Output directory: {OUTPUT_DIR}")


Configuration loaded.
Number of source files: 21
Output directory: model_datasets\stage_2_9pct


In [2]:

original_train = pd.read_parquet(
    PREVIOUS_DATA_DIR / "train.parquet"
)

original_validation = pd.read_parquet(
    PREVIOUS_DATA_DIR / "validation.parquet"
)

original_sampled = pd.read_parquet(
    PREVIOUS_DATA_DIR / "sampled_5pct.parquet"
)

print("Original dataset loaded.")

print(f"Original training spectra: {len(original_train):,}")
print(f"Original validation spectra: {len(original_validation):,}")
print(f"Original sampled spectra: {len(original_sampled):,}")

# Check required columns
required_columns = {"row_id", "inchikey14", "source_part"}

for name, df in [
    ("Original training", original_train),
    ("Original validation", original_validation),
    ("Original sampled", original_sampled)
]:
    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(
            f"{name} is missing required columns: {missing}"
        )

print("\nRequired columns verified.")


Original dataset loaded.
Original training spectra: 91,851
Original validation spectra: 22,963
Original sampled spectra: 114,814

Required columns verified.


In [3]:

# Create unique identifiers for all previously sampled spectra

previously_sampled_keys = set(
    zip(
        original_sampled["source_part"].astype(str),
        original_sampled["row_id"].astype(str)
    )
)

print(
    f"Previously sampled spectra to exclude: "
    f"{len(previously_sampled_keys):,}"
)


Previously sampled spectra to exclude: 114,814


In [4]:

additional_samples = []
additional_train_parts = []

for part_number, filename in enumerate(PART_FILES):

    print(f"\nProcessing {filename}...")

    file_path = SPECTRA_DIR / filename

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    # Load original parquet file
    part_df = pd.read_parquet(file_path)

    # Record source file
    part_df["source_part"] = filename.replace(".parquet", "")

    # Identify previously sampled rows
    current_keys = list(
        zip(
            part_df["source_part"].astype(str),
            part_df["row_id"].astype(str)
        )
    )

    already_sampled_mask = np.array(
        [
            key in previously_sampled_keys
            for key in current_keys
        ],
        dtype=bool
    )

   # Remove previously sampled rows
    remaining_df = part_df.loc[
        ~already_sampled_mask
    ].copy()

    # Sample 5% of remaining data
    n_additional = round(
        len(remaining_df) * ADDITIONAL_SAMPLE_FRACTION
    )

    additional_sample = remaining_df.sample(
        n=n_additional,
        random_state=SEED + part_number
    ).copy()

    # Shuffle the additional sample
    additional_sample = additional_sample.sample(
        frac=1,
        random_state=SEED + 1000 + part_number
    ).reset_index(drop=True)

    # Use 80% for additional training
    n_train = round(
        len(additional_sample) * TRAIN_FRACTION
    )

    additional_train = additional_sample.iloc[:n_train].copy()

    # Store results
    additional_samples.append(additional_sample)
    additional_train_parts.append(additional_train)

    print(f"  Original rows: {len(part_df):,}")
    print(f"  Previously sampled: {sum(already_sampled_mask):,}")
    print(f"  Remaining rows: {len(remaining_df):,}")
    print(f"  Additional sample: {len(additional_sample):,}")
    print(f"  Additional training: {len(additional_train):,}")
    print(f"  Unused remainder: {len(additional_sample) - len(additional_train):,}")

# Combine additional samples
additional_sampled_df = pd.concat(
    additional_samples,
    ignore_index=True
)

additional_train_df = pd.concat(
    additional_train_parts,
    ignore_index=True
)

print("\nAdditional sampling completed.")
print(f"Total additional sample: {len(additional_sampled_df):,}")
print(f"Total additional training: {len(additional_train_df):,}")



Processing part-000.parquet...
  Original rows: 126,233
  Previously sampled: 0
  Remaining rows: 126,233
  Additional sample: 6,312
  Additional training: 5,050
  Unused remainder: 1,262

Processing part-001.parquet...
  Original rows: 126,164
  Previously sampled: 0
  Remaining rows: 126,164
  Additional sample: 6,308
  Additional training: 5,046
  Unused remainder: 1,262

Processing part-002.parquet...
  Original rows: 126,305
  Previously sampled: 0
  Remaining rows: 126,305
  Additional sample: 6,315
  Additional training: 5,052
  Unused remainder: 1,263

Processing part-003.parquet...
  Original rows: 126,606
  Previously sampled: 0
  Remaining rows: 126,606
  Additional sample: 6,330
  Additional training: 5,064
  Unused remainder: 1,266

Processing part-004.parquet...
  Original rows: 126,387
  Previously sampled: 0
  Remaining rows: 126,387
  Additional sample: 6,319
  Additional training: 5,055
  Unused remainder: 1,264

Processing part-005.parquet...
  Original rows: 126,43

In [5]:

# Combine original training data with additional training data

combined_train_df = pd.concat(
    [original_train, additional_train_df],
    ignore_index=True
)

# Keep original validation data unchanged
combined_validation_df = original_validation.copy()

print("Combined dataset created.")

print(f"\nOriginal training data: {len(original_train):,}")
print(f"Additional training data: {len(additional_train_df):,}")
print(f"Combined training data: {len(combined_train_df):,}")
print(f"Validation data: {len(combined_validation_df):,}")

print(
    f"\nTraining data as percentage of original sample:"
    f" {len(combined_train_df) / (len(original_train) + len(original_validation)) * 100:.2f}%"
)


Combined dataset created.

Original training data: 91,851
Additional training data: 91,851
Combined training data: 183,702
Validation data: 22,963

Training data as percentage of original sample: 160.00%


In [6]:

def get_row_keys(df):
    return set(
        zip(
            df["source_part"].astype(str),
            df["row_id"].astype(str)
        )
    )

original_keys = get_row_keys(original_sampled)
additional_keys = get_row_keys(additional_sampled_df)

original_train_keys = get_row_keys(original_train)
additional_train_keys = get_row_keys(additional_train_df)
validation_keys = get_row_keys(combined_validation_df)

# Check overlap
overlap_with_original = original_keys.intersection(additional_keys)

overlap_train_validation = (
    get_row_keys(combined_train_df).intersection(validation_keys)
)

# Check duplicates
duplicate_combined_train = combined_train_df.duplicated(
    subset=["source_part", "row_id"]
).sum()

print("DATA INTEGRITY CHECKS")
print("---------------------")

print(
    f"Overlap with original sample: "
    f"{len(overlap_with_original):,}"
)

print(
    f"Training/validation overlap: "
    f"{len(overlap_train_validation):,}"
)

print(
    f"Duplicate rows in combined training: "
    f"{duplicate_combined_train:,}"
)

assert len(overlap_with_original) == 0
assert len(overlap_train_validation) == 0
assert duplicate_combined_train == 0

print("\nAll integrity checks passed.")


DATA INTEGRITY CHECKS
---------------------
Overlap with original sample: 0
Training/validation overlap: 0
Duplicate rows in combined training: 0

All integrity checks passed.


In [7]:

# Load original class mapping
with open(
    PREVIOUS_DATA_DIR / "class_mapping.json",
    "r"
) as f:
    original_mapping_json = json.load(f)


def extract_class_mapping(data):

    # Direct class-to-index mapping
    if all(
        isinstance(v, int)
        for v in data.values()
    ):
        return data

    # Common nested formats
    for key in ["class_to_idx", "class_mapping"]:
        if key in data and isinstance(data[key], dict):
            return data[key]

    # Index-to-class mapping
    if "idx_to_class" in data:
        return {
            label: int(index)
            for index, label in data["idx_to_class"].items()
        }

    raise ValueError(
        "Could not identify class-to-index mapping format."
    )


original_class_to_idx = extract_class_mapping(
    original_mapping_json
)

# Normalise labels to strings
original_class_to_idx = {
    str(label): int(index)
    for label, index in original_class_to_idx.items()
}

# Identify all classes in combined training data
combined_classes = set(
    combined_train_df["inchikey14"].dropna().astype(str)
)

# Preserve original indices
updated_class_to_idx = original_class_to_idx.copy()

next_index = max(updated_class_to_idx.values()) + 1

# Append new classes
new_classes = sorted(
    combined_classes - set(updated_class_to_idx.keys())
)

for label in new_classes:
    updated_class_to_idx[label] = next_index
    next_index += 1

print(f"Original classes: {len(original_class_to_idx):,}")
print(f"New classes introduced: {len(new_classes):,}")
print(f"Updated total classes: {len(updated_class_to_idx):,}")

# Ensure all validation classes are represented
validation_classes = set(
    combined_validation_df["inchikey14"].dropna().astype(str)
)

unmapped_validation = (
    validation_classes - set(updated_class_to_idx.keys())
)

print(
    f"Validation classes not represented in training: "
    f"{len(unmapped_validation):,}"
)

print("\nUpdated training class mapping verified.")


Original classes: 65,644
New classes introduced: 40,066
Updated total classes: 105,710
Validation classes not represented in training: 8,167

Updated training class mapping verified.


In [8]:

# Original sample manifest
original_manifest = original_sampled[
    ["source_part", "row_id", "inchikey14"]
].copy()

original_train_key_set = get_row_keys(original_train)

original_manifest["split"] = [
    "train" if key in original_train_key_set else "validation"
    for key in zip(
        original_manifest["source_part"].astype(str),
        original_manifest["row_id"].astype(str)
    )
]

# Additional training manifest
additional_manifest = additional_train_df[
    ["source_part", "row_id", "inchikey14"]
].copy()

additional_manifest["split"] = "additional_train"

# Combine manifests
split_manifest = pd.concat(
    [original_manifest, additional_manifest],
    ignore_index=True
)

print("Combined split manifest:")
print(split_manifest["split"].value_counts())

print(f"\nTotal manifest rows: {len(split_manifest):,}")


Combined split manifest:
split
train               91851
additional_train    91851
validation          22963
Name: count, dtype: int64

Total manifest rows: 206,665


In [9]:

# Save combined training data
combined_train_df.to_parquet(
    OUTPUT_DIR / "train.parquet",
    index=False
)

# Save original validation data unchanged
combined_validation_df.to_parquet(
    OUTPUT_DIR / "validation.parquet",
    index=False
)

# Save additional sample and additional training data
additional_sampled_df.to_parquet(
    OUTPUT_DIR / "additional_sampled.parquet",
    index=False
)

additional_train_df.to_parquet(
    OUTPUT_DIR / "additional_train.parquet",
    index=False
)

# Save combined manifest
split_manifest.to_csv(
    OUTPUT_DIR / "split_manifest.csv",
    index=False
)

# Save updated class mapping
with open(
    OUTPUT_DIR / "class_mapping.json",
    "w"
) as f:
    json.dump(
        updated_class_to_idx,
        f,
        indent=2
    )

print("Experiment 3 datasets saved.")



Experiment 3 datasets saved.


In [10]:


experiment_config = {
    "experiment": "Fine-tuned DreaMS classification - Experiment 3",
    "seed": SEED,
    "original_sample_fraction_per_part": 0.05,
    "additional_sample_fraction_of_remaining_per_part": 0.05,
    "additional_train_fraction": 0.80,
    "original_training_spectra": len(original_train),
    "additional_training_spectra": len(additional_train_df),
    "combined_training_spectra": len(combined_train_df),
    "validation_spectra": len(combined_validation_df),
    "original_training_classes": len(original_class_to_idx),
    "updated_training_classes": len(updated_class_to_idx),
    "validation_source": "Original Experiment 1 validation set",
    "validation_filtering": "Applied during model training using known training classes",
    "heldout_data_used": False
}

with open(
    OUTPUT_DIR / "experiment_config.json",
    "w"
) as f:
    json.dump(
        experiment_config,
        f,
        indent=2
    )

print("Experiment configuration saved.")



Experiment configuration saved.


In [11]:

print("=" * 55)
print("EXPERIMENT 3 DATASET SUMMARY")
print("=" * 55)

print(f"Combined training spectra: {len(combined_train_df):,}")
print(f"Validation spectra: {len(combined_validation_df):,}")
print(f"Total classes: {len(updated_class_to_idx):,}")

print("\nSaved files:")

for file in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {file.name}")

print("\nDataset preparation completed successfully.")


EXPERIMENT 3 DATASET SUMMARY
Combined training spectra: 183,702
Validation spectra: 22,963
Total classes: 105,710

Saved files:
  additional_sampled.parquet
  additional_train.parquet
  class_mapping.json
  experiment_config.json
  split_manifest.csv
  train.parquet
  validation.parquet

Dataset preparation completed successfully.
